# Calculate monthly PM2.5 using eq. from [Turnock et al. (2022)](https://doi.org/10.1029/2022EF002687)

Since there are some models which do not output PM2.5 as a singular variable we calculate monthly PM2.5 using a combination of mass mixing ratios and converting to a concentration: 

PM2.5 = BC + OA + SO4 + (0.2xSS) + (0.1xDU)

This script is an example of processing CESM2 data that is readily available from the model into the format needed to begin the common processing steps throughout the rest of the workflow - monthly PM2.5 in μg/m3.

In [ ]:
import os
import re
import glob
import warnings
import xarray as xr
from collections import defaultdict
from utils.utils import get_scenario_config, load_file_list, minus_one_month

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/file_paths/"
SCRATCH = f"/glade/derecho/scratch/awells/air_quality/{model}/pm25/"
T_DIR = f"/glade/work/awells/air_quality/{model}/temp/temp_pres/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/monthly_pm25/"

VAR_list = ["BC", "POA", "SOA", "SO4", "SS", "DU"]

In [ ]:
def load_file(f):
    if not os.path.exists(f):
        raise ValueError(f"Missing: {f}")

    # Find variable
    pattern = re.compile(r"cam\.h0\.([^.]+)\.")
    m = pattern.search(f)
    varname = m.group(1)

    ds = xr.open_dataset(f)
    da = ds[varname]
    return da


def select_surface(da):
    return da.isel(lev=-1)

For mass mixing ratios of small particles in CESM2, the variables are usually separated by mode - for example, the so4 variables are called so4_a1, so4_a2, so4_a3, so4_c1, so4_c2, and so4_c3, where 1-3 are accumulation, aitken, and coarse modes, respectively, and "a" variables are for dry particles and "c" are for in-cloud. Likewise, for BC, you can look for bc_a1, bc_a2, etc.

Read more [here](https://wiki.ucar.edu/spaces/camchem/pages/358319530/Aerosols)

This section of code calculates the total sum for each variable.

In [ ]:
for var in VAR_list:
    for ens_num in ensemble_members:
        print(f"Processing {var}, {scenario} ensemble {ens_num:02d}")
        # Load all file lists
        file_list = load_file_list(FILE_DIR, f"file_list_{var}_{scenario}_{ens_num:02d}.json")

        groups = defaultdict(list)
        for f in file_list:
            # Load surface variable
            da = select_surface(load_file(f))
            groups[da.name].append(da)

        combined = {}
        for vbl, das in groups.items():
            combined[vbl] = xr.concat(
                das,
                dim="time",
                join="outer",
                combine_attrs="drop_conflicts"
            )

        # Get unit attribute from combined
        first_key = next(iter(combined))
        units = combined[first_key].attrs["units"]

        # Calculate the sum of all modes etc.
        total_var = sum(combined.values())
        total_var.attrs["units"] = units

        # If the first month is February (2) then apply month fixer
        first_month = total_var.time.dt.month[0]
        if first_month == 2:
            print("Adjusting month indexing")
            new_time = [minus_one_month(t) for t in total_var["time"].values]
            total_var = total_var.assign_coords(time=new_time)
        # If the first month is January (1) don't apply month fixer
        elif first_month == 1:
            print("No month index adjusting needed")
        else:
            warnings.warn(f"First month: {first_month}, check dates in file")

        first_year = total_var.time.dt.year[0].item()
        last_year = total_var.time.dt.year[-1].item()

        out_file = f"{var}_mmr_{model}_{scenario}_{ens_num:02d}_{first_year}-{last_year}.nc"
        out_path = os.path.join(SCRATCH, out_file)
        total_var.to_netcdf(out_path)

print("All processing complete.")

The conversion of mmr to concentration requires calculating air density from pressure and temperature

In [ ]:
def mmr_to_conc(mmr, p_hPa, T):
    """
    Convert mmr (kg/kg) to concentration (µg/m³)
    calculating air density from pressure and temperature
    """
    # convert pressure to Pa
    p = p_hPa * 100.0

    T = T.sel(lev=p_hPa)

    # compute density
    Rd = 287.0  # J/kg/K
    rho = p / (Rd * T)

    # convert mixing ratio to μg/m3
    conc = mmr * rho * 1e9
    conc.attrs["units"] = "μg/m3"
    return conc

In [ ]:
def load_var(var):
    pattern = f"{var}_mmr_{model}_{scenario}_{ens_num:02d}_*.nc"
    path = glob.glob(os.path.join(SCRATCH, pattern))[0]
    return xr.open_dataarray(path, chunks={"time": 1})  # or a chunk size that fits memory


for ens_num in ensemble_members:
    print(f"Calculating PM2.5, {scenario} ensemble {ens_num:02d}")

    # lazy-loaded & chunked variables (NO memory explosion)
    poa = load_var("POA")
    soa = load_var("SOA")
    bc = load_var("BC")
    so4 = load_var("SO4")
    ss = load_var("SS")
    du = load_var("DU")

    # Compute OA lazily
    oa = poa + soa

    # Compute PM2.5
    pm25 = bc + oa + so4 + (0.2 * ss) + (0.1 * du)

    # Load temperature
    T_pattern = f"T_{model}_{scenario}_{ens_num:02d}_*.nc"
    T_path = glob.glob(os.path.join(T_DIR, T_pattern))[0]
    T = xr.open_dataarray(T_path, chunks={"time": 1})

    # Convert to concentration
    pm25_conc = mmr_to_conc(pm25, pm25.lev, T)
    # Slice time to match years for each scenario
    sliced_da = pm25_conc.sel(time=slice(str(years.start), str(years.stop)))
    # Remove the unused lev dimension
    sliced_da = sliced_da.drop_vars("lev")

    dates = f"{years.start}01-{years.stop}12"

    description = ("Calculated monthly surface PM2.5 using equation from "
                   "Turnock et al. (2022) - scripts by A.F. Wells (2025)")
    sliced_da.attrs["description"] = description
    sliced_da.attrs["ensemble_number"] = ens_num
    sliced_da.attrs["scenario"] = scenario
    sliced_da.attrs["model"] = model

    # Only write to disk (still lazy)
    out_file = f"Monthly_PM25_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    sliced_da.to_netcdf(out_path, compute=True, engine="netcdf4")

print("All processing complete.")